— Fusion des tables
Objectif : produire 2 tables finales
 → technicians_final.csv
 → woo_final.csv


In [1]:


import pandas as pd
import os

# On lit depuis les CSV PROPRES (résultat du nettoyage Phase 1)
INPUT_DIR  = "./clean_tables"   # ← vos CSV nettoyés
OUTPUT_DIR = "./final_tables"   # ← les 2 tables finales

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f" Configuration OK")
print(f"   Lecture depuis : {os.path.abspath(INPUT_DIR)}")
print(f"   Écriture vers  : {os.path.abspath(OUTPUT_DIR)}")

# ── Chargement des CSV propres ────────────────────────────────────
df_availability = pd.read_csv(f"{INPUT_DIR}/availability.csv",            encoding="utf-8-sig")
df_customer     = pd.read_csv(f"{INPUT_DIR}/customer.csv",                encoding="utf-8-sig")
df_skills       = pd.read_csv(f"{INPUT_DIR}/skills.csv",                  encoding="utf-8-sig")
df_technicians  = pd.read_csv(f"{INPUT_DIR}/technicians.csv",             encoding="utf-8-sig")
df_work_centers = pd.read_csv(f"{INPUT_DIR}/work_centers.csv",            encoding="utf-8-sig")
df_woo          = pd.read_csv(f"{INPUT_DIR}/work_order_operations.csv",   encoding="utf-8-sig")
df_assets       = pd.read_csv(f"{INPUT_DIR}/assets.csv",                  encoding="utf-8-sig")
df_operations   = pd.read_csv(f"{INPUT_DIR}/operations.csv",              encoding="utf-8-sig")
df_asset_crit   = pd.read_csv(f"{INPUT_DIR}/asset_criticality.csv",       encoding="utf-8-sig")

# ── Résumé du chargement ─────────────────────────────────────────
tables = {
    "availability"   : df_availability,
    "customer"       : df_customer,
    "skills"         : df_skills,
    "technicians"    : df_technicians,
    "work_centers"   : df_work_centers,
    "woo"            : df_woo,
    "assets"         : df_assets,
    "operations"     : df_operations,
    "asset_crit"     : df_asset_crit,
}

print(f"\n{'Table':<20} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 40)
for name, df in tables.items():
    print(f"{name:<20} {len(df):>8} {len(df.columns):>10}")




 Configuration OK
   Lecture depuis : c:\Maria\digitalPlanner\clean_tables
   Écriture vers  : c:\Maria\digitalPlanner\final_tables

Table                  Lignes   Colonnes
----------------------------------------
availability                2          7
customer                   62          3
skills                      4          2
technicians                 7         12
work_centers                3          3
woo                      1096         22
assets                    145         12
operations                 27          9
asset_crit                  3          1



Construction table TECHNICIANS fusionnée

 enrichir la table technicians avec les infos de  Skills, Availability et Work_Centers

JOINTURES :
 technicians + skills       → via fk_skill_id
 technicians + availability → via fk_availability_id
 technicians + work_centers → via fk_work_center_id

 TYPE DE JOINTURE : LEFT JOIN
 → on garde TOUS les techniciens même si une info est manquante



In [3]:
# ── Étape 1 : Technicians + Skills 
 #On ajoute skill_description depuis la table skills
 #fk_skill_id dans technicians = pk_skill_id dans skills


df_tech_final = df_technicians.merge(
    df_skills[["pk_skill_id", "skill_description"]],
    left_on  = "fk_skill_id",
    right_on = "pk_skill_id",
    how      = "left"
)

# pk_skill_id est devenu inutile après la jointure (c'est une copie de fk_skill_id)
df_tech_final = df_tech_final.drop(columns=["pk_skill_id"])


print(": Technicians + Skills ")
print(f"Colonnes après fusion : {len(df_tech_final.columns)}")
print(df_tech_final[["technician_full_name", "fk_skill_id", "skill_description"]].to_string())

# ── Étape 2 : Technicians+ Availability 
# On ajoute status, start_hour, end_hour, pause_start, pause_end
# fk_availability_id dans technicians = pk_working_hour_id dans availability

df_tech_final = df_tech_final.merge(
    df_availability[[
        "pk_working_hour_id",
        "status",
        "working_hour_description",
        "start_hour",
        "end_hour",
        "pause_start",
        "pause_end"
    ]],
    left_on  = "fk_availability_id",
    right_on = "pk_working_hour_id",
    how      = "left"
)

df_tech_final = df_tech_final.drop(columns=["pk_working_hour_id"])

df_tech_final = df_tech_final.rename(columns={"status": "availability_status"})


print("\n── Étape 2 : + Availability ")
print(f"Colonnes après fusion : {len(df_tech_final.columns)}")
print(df_tech_final[["technician_full_name", "availability_status", "start_hour", "end_hour", "pause_start", "pause_end"]].to_string())

# ── Étape 3 :Technicians + Work_Centers 
# On ajoute work_center_description et work_center_status,
# fk_work_center_id dans technicians = pk_work_center_id dans work_centers

df_tech_final = df_tech_final.merge(
    df_work_centers[["pk_work_center_id", "work_center_status", "work_center_description"]],
    left_on  = "fk_work_center_id",
    right_on = "pk_work_center_id",
    how      = "left"
)

df_tech_final = df_tech_final.drop(columns=["pk_work_center_id"])

print("\n── Étape 3 : + Work_Centers ────────────────────────────────")
print(f"Colonnes après fusion : {len(df_tech_final.columns)}")
print(df_tech_final[["technician_full_name", "fk_work_center_id", "work_center_status", "work_center_description"]].to_string())

# ── Résumé ───────────────────────────────────────────────────────
print(f"\n── Résumé table TECHNICIANS fusionnée ──────────────────────")
print(f"Lignes   : {len(df_tech_final)}")
print(f"Colonnes : {len(df_tech_final.columns)}")
print(f"\nToutes les colonnes :")
for col in df_tech_final.columns:
    non_null = df_tech_final[col].notna().sum()
    print(f"   {'✅' if non_null > 0 else '⬜'} {col:<35} {non_null}/{len(df_tech_final)} valeurs remplies")


: Technicians + Skills 
Colonnes après fusion : 13
  technician_full_name  fk_skill_id skill_description
0           Luc Dupuis            2        Mecanician
1       André Philippe            3  Inspector Junior
2      Charles Renault            4  Inspector Senior
3      Jacques Lefèvre            1       Electrician
4          Jean Durand            3  Inspector Junior
5       Mathieu Gérard            3  Inspector Junior
6         Victor Caron            4  Inspector Senior

── Étape 2 : + Availability 
Colonnes après fusion : 19
  technician_full_name availability_status start_hour end_hour pause_start pause_end
0           Luc Dupuis              Active       7:00    17:00       11:00     12:00
1       André Philippe              Active       7:00    17:00       11:00     12:00
2      Charles Renault              Active       7:00    17:00       11:00     12:00
3      Jacques Lefèvre              Active       7:00    17:00       11:00     12:00
4          Jean Durand             

 Nettoyage colonnes TECHNICIANS fusionnée

 Après les jointures, certaines colonnes sont devenues inutiles :

fk_skill_id        → on a maintenant skill_description directement
fk_availability_id → on a maintenant start_hour, end_hour, pause...
fk_work_center_id  → on a maintenant work_center_description

Ces colonnes FK étaient utiles pour faire les jointures mais dans une table plate finale elles n'apportent rien

In [4]:


cols_to_drop = [
    "fk_skill_id",          # remplacé par skill_description
    "fk_availability_id",   # remplacé par start_hour, end_hour, pause_start, pause_end
    "fk_work_center_id",    # remplacé par work_center_description
]

df_tech_final = df_tech_final.drop(columns=cols_to_drop)

# ── Réorganisation des colonnes dans un ordre logique ────────────
# POURQUOI ?
# Pour qu'Excel et Airtable affichent les colonnes dans un ordre
# cohérent et lisible — infos technicien d'abord, puis compétences,
# puis disponibilités, puis localisation

cols_ordered = [
    # Identité du technicien
    "pk_technician_id",
    "technician_status",
    "technician_first_name",
    "technician_last_name",
    "technician_full_name",
    # Compétence
    "skill_description",
    # Disponibilité
    "availability_status",
    "working_hour_description",
    "start_hour",
    "end_hour",
    "pause_start",
    "pause_end",
    # Localisation technicien
    "address_street",
    "address_door",
    "address_post_code",
    "address_city",
    # Centre de travail
    "work_center_status",
    "work_center_description",
]

df_tech_final = df_tech_final[cols_ordered]

# ── Résumé final 
print("── Table TECHNICIANS finale ")
print(f"Lignes   : {len(df_tech_final)}")
print(f"Colonnes : {len(df_tech_final.columns)}")
print(f"\nColonnes finales dans l'ordre :")
for col in df_tech_final.columns:
    non_null = df_tech_final[col].notna().sum()
    print(f"   ✅ {col:<35} {non_null}/{len(df_tech_final)} valeurs remplies")

print(f"\nAperçu de la table finale :")
print(df_tech_final.to_string())

── Table TECHNICIANS finale 
Lignes   : 7
Colonnes : 18

Colonnes finales dans l'ordre :
   ✅ pk_technician_id                    7/7 valeurs remplies
   ✅ technician_status                   7/7 valeurs remplies
   ✅ technician_first_name               7/7 valeurs remplies
   ✅ technician_last_name                7/7 valeurs remplies
   ✅ technician_full_name                7/7 valeurs remplies
   ✅ skill_description                   7/7 valeurs remplies
   ✅ availability_status                 7/7 valeurs remplies
   ✅ working_hour_description            7/7 valeurs remplies
   ✅ start_hour                          7/7 valeurs remplies
   ✅ end_hour                            7/7 valeurs remplies
   ✅ pause_start                         7/7 valeurs remplies
   ✅ pause_end                           7/7 valeurs remplies
   ✅ address_street                      7/7 valeurs remplies
   ✅ address_door                        7/7 valeurs remplies
   ✅ address_post_code                   7/

— Construction table WOO fusionnée
 

 OBJECTIF : enrichir WOO avec les infos de
 Assets, Customer, Asset_Criticality

 JOINTURES :
 woo + assets          → via fk_asset_id
 assets + customer     → via fk_customer_id
 assets + asset_crit   → via fk_criticality

 TYPE : LEFT JOIN → on garde toutes les opérations
 même les 34 qui n'ont pas d'asset identifié (fk_asset_id NULL)

 

In [5]:

#  WOO + Assets 
# On récupère toutes les infos utiles de l'asset
# L'algorithme a besoin de savoir :
# - Où se trouve l'asset (adresse) → pour calculer les trajets
# - Quelle est sa criticité        → pour prioriser
# - Quel client                    → pour le reporting

df_woo_final = df_woo.merge(
    df_assets[[
        "pk_asset_id",
        "asset_status",
        "asset_description",
        "address_street",
        "address_door",
        "address_post_code",
        "address_city",
        "asset_region",
        "fk_customer_id",
        "fk_criticality",
        "created_at"
    ]],
    left_on  = "fk_asset_id",
    right_on = "pk_asset_id",
    how      = "left"
)
df_woo_final = df_woo_final.drop(columns=["pk_asset_id"])

# Renommer les colonnes d'adresse pour éviter confusion avec adresse technicien
df_woo_final = df_woo_final.rename(columns={
    "address_street"    : "asset_address_street",
    "address_door"      : "asset_address_door",
    "address_post_code" : "asset_address_post_code",
    "address_city"      : "asset_address_city",
    "asset_region"      : "asset_address_region",
    "created_at"        : "asset_created_at"
})

print("── Étape 1 : WOO + Assets ")
print(f"Lignes   : {len(df_woo_final)}")
print(f"Colonnes : {len(df_woo_final.columns)}")
print(f"Assets identifiés   : {df_woo_final['fk_asset_id'].notna().sum()} / {len(df_woo_final)}")
print(f"Assets NULL (34 WO) : {df_woo_final['fk_asset_id'].isna().sum()} / {len(df_woo_final)}")

# WOO + Customer 
# On récupère customer_description

df_woo_final = df_woo_final.merge(
    df_customer[["pk_customer_id", "customer_description"]],
    left_on  = "fk_customer_id",
    right_on = "pk_customer_id",
    how      = "left"
)
df_woo_final = df_woo_final.drop(columns=["pk_customer_id"])

print("\n── Étape 2 : WOO + Customer ")
print(f"Colonnes : {len(df_woo_final.columns)}")
print(df_woo_final[["pk_woo_id", "asset_description", "customer_description"]].head(5).to_string())

# ── Étape 3 : WOO + Asset_Criticality ────────────────────────────────
# fk_criticality dans assets = pk_criticality_name dans asset_criticality
# Valeurs : Good / Adequate / Bad

# La criticité de l'asset influence la priorité de l'intervention
# Un asset "Bad" doit être traité plus rapidement

# asset_criticality n'a qu'une seule colonne pk_criticality_name
# elle est déjà dans df_woo_final via fk_criticality
# on la renomme simplement pour plus de clarté

df_woo_final = df_woo_final.rename(columns={
    "fk_criticality" : "asset_criticality"
})

print("\n── Étape 3 : Asset Criticality ──────────────────────────────")
print(f"Colonnes : {len(df_woo_final.columns)}")
print(f"Répartition criticité :")
print(df_woo_final["asset_criticality"].value_counts().to_string())

# ── Étape 4 : WOO + Operations ───────────────────────────────────────

# La table Operations contient des infos sur le template de l'opération
# que WOO n'a pas toujours :
#   - skills_possibilities → quelles compétences peuvent faire cette op
#   - predecessor_operation_key → ordre d'exécution des opérations
#
# fk_operation_id dans WOO = pk_operation_id dans Operations

df_woo_final = df_woo_final.merge(
    df_operations[[
        "pk_operation_id",
        "operation_type",
        "operation_subtype",
        "operation_order",
        "predecessor_operation_key",
        "duration",
        "duration_unit"
    ]],
    left_on  = "fk_operation_id",
    right_on = "pk_operation_id",
    how      = "left",
    suffixes = ("", "_from_operations")   # évite les conflits de noms
)
df_woo_final = df_woo_final.drop(columns=["pk_operation_id"])

# Vérifier les colonnes en doublon après jointure
# duration, duration_unit, operation_type, operation_subtype, operation_order
# existent déjà dans WOO — on garde celles de WOO et supprime celles d'Operations
cols_duplicate = [
    "operation_type_from_operations",
    "operation_subtype_from_operations",
    "operation_order_from_operations",
    "duration_from_operations",
    "duration_unit_from_operations",
    "predecessor_operation_key_from_operations"
]

# On supprime uniquement celles qui existent
cols_to_drop = [c for c in cols_duplicate if c in df_woo_final.columns]
df_woo_final = df_woo_final.drop(columns=cols_to_drop)

print("\n── Étape 4 : WOO+ Operations ")
print(f"Colonnes : {len(df_woo_final.columns)}")
print(f"Correspondances WOO → Operations : {df_woo_final['fk_operation_id'].notna().sum()} / {len(df_woo_final)}")

# ── Résumé 
print(f"\n── Résumé table WOO fusionnée ")
print(f"Lignes   : {len(df_woo_final)}")
print(f"Colonnes : {len(df_woo_final.columns)}")
print(f"\nToutes les colonnes :")
for col in df_woo_final.columns:
    non_null = df_woo_final[col].notna().sum()
    print(f"   {'✅' if non_null > 0 else '⬜'} {col:<40} {non_null}/{len(df_woo_final)} valeurs remplies")

── Étape 1 : WOO + Assets 
Lignes   : 1096
Colonnes : 32
Assets identifiés   : 983 / 1096
Assets NULL (34 WO) : 113 / 1096

── Étape 2 : WOO + Customer 
Colonnes : 33
   pk_woo_id    asset_description customer_description
0        132  Asset Description 1           Customer 1
1        133  Asset Description 1           Customer 1
2        134  Asset Description 1           Customer 1
3        135  Asset Description 1           Customer 1
4        136  Asset Description 1           Customer 1

── Étape 3 : Asset Criticality ──────────────────────────────
Colonnes : 33
Répartition criticité :
asset_criticality
Good        352
Adequate    347
Bad         284

── Étape 4 : WOO+ Operations 
Colonnes : 33
Correspondances WOO → Operations : 1096 / 1096

── Résumé table WOO fusionnée 
Lignes   : 1096
Colonnes : 33

Toutes les colonnes :
   ✅ pk_woo_id                                1096/1096 valeurs remplies
   ✅ status                                   1096/1096 valeurs remplies
   ✅ fk_opera


— Nettoyage colonnes WOO fusionnée

 Après les jointures, certaines colonnes FK sont devenues inutiles
 car on a maintenant les vraies valeurs directement.
 On supprime aussi les colonnes peu utiles pour le client
 et on réorganise dans un ordre logique.

In [6]:


#  Supprimer les colonnes FK devenues inutiles ────────
cols_to_drop = [
    "fk_operation_id",          # on a operation_key + description directement
    "fk_required_skill_id",     # on a required_skill_desc directement
    "fk_asset_id",              # on a asset_description + adresse directement
    "fk_customer_id",           # on a customer_description directement
    "asset_created_at",         # info technique inutile pour le planning
]

df_woo_final = df_woo_final.drop(columns=cols_to_drop)

print("── Étape 1 : Colonnes FK supprimées ")
print(f"Colonnes restantes : {len(df_woo_final.columns)}")

#   Réorganisation dans un ordre logique ───────────────
# Pour qu'Excel et Airtable affichent les colonnes dans un ordre
# cohérent et lisible pour le Planning Officer

cols_ordered = [
    # Identification de l'opération
    "pk_woo_id",
    "status",
    "operation_key",
    "operation_description",
    "operation_type",
    "operation_subtype",
    "operation_order",
    "priority_score",

    # Durée
    "duration",
    "duration_unit",
    "confirmed_work",
    "confirmed_work_unit",

    # Prérequis
    "required_skill_desc",
    "predecessor_operation_key",

    # Dates souhaitées
    "order_basic_start_date",
    "order_basic_end_date",

    # Planification (remplies par le planner)
    "operation_scheduled_start",
    "operation_scheduled_end",

    # Technicien assigné (rempli par le planner)
    "fk_assigned_technician_id",

    # Asset concerné
    "asset_status",
    "asset_description",
    "asset_criticality",
    "asset_address_street",
    "asset_address_door",
    "asset_address_post_code",
    "asset_address_city",
    "asset_address_region",

    # Client
    "customer_description",
]

df_woo_final = df_woo_final[cols_ordered]

# ── Résumé final 
print("\n── Table WOO finale ")
print(f"Lignes   : {len(df_woo_final)}")
print(f"Colonnes : {len(df_woo_final.columns)}")
print(f"\nColonnes finales dans l'ordre :")
for col in df_woo_final.columns:
    non_null = df_woo_final[col].notna().sum()
    statut   = "✅" if non_null > 0 else "⬜"
    note     = " ← rempli par le planner" if col in [
        "operation_scheduled_start",
        "operation_scheduled_end",
        "fk_assigned_technician_id"
    ] else ""
    print(f"   {statut} {col:<40} {non_null}/{len(df_woo_final)} valeurs remplies{note}")

print(f"\nAperçu des 3 premières lignes :")
print(df_woo_final.head(3).to_string())

── Étape 1 : Colonnes FK supprimées 
Colonnes restantes : 28

── Table WOO finale 
Lignes   : 1096
Colonnes : 28

Colonnes finales dans l'ordre :
   ✅ pk_woo_id                                1096/1096 valeurs remplies
   ✅ status                                   1096/1096 valeurs remplies
   ✅ operation_key                            1096/1096 valeurs remplies
   ✅ operation_description                    1096/1096 valeurs remplies
   ✅ operation_type                           1096/1096 valeurs remplies
   ✅ operation_subtype                        1096/1096 valeurs remplies
   ✅ operation_order                          1096/1096 valeurs remplies
   ✅ priority_score                           1096/1096 valeurs remplies
   ✅ duration                                 1096/1096 valeurs remplies
   ✅ duration_unit                            1096/1096 valeurs remplies
   ✅ confirmed_work                           1096/1096 valeurs remplies
   ✅ confirmed_work_unit                      1096/

— Vérification finale des 2 tables
Avant de sauvegarder, on vérifie que les 2 tables sont cohérentes
entre elles — notamment que les techniciens référencés dans WOO
existent bien dans la table TECHNICIANS

In [7]:

print("  VÉRIFICATION FINALE DES 2 TABLES")

# ── Résumé général ───────────────────────────────────────────────
print(f"\n{'Table':<25} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 45)
print(f"{'technicians_final':<25} {len(df_tech_final):>8} {len(df_tech_final.columns):>10}")
print(f"{'woo_final':<25} {len(df_woo_final):>8} {len(df_woo_final.columns):>10}")

# ── Vérification 1 : colonnes clés présentes ─────────────────────
print("\n── Vérification 1 : colonnes clés présentes ────────────────")

cols_required_tech = [
    "pk_technician_id", "technician_full_name", "skill_description",
    "start_hour", "end_hour", "work_center_description"
]
cols_required_woo = [
    "pk_woo_id", "priority_score", "operation_type", "operation_subtype",
    "required_skill_desc", "operation_scheduled_start", "operation_scheduled_end",
    "fk_assigned_technician_id"
]

print("\nTECHNICIANS — colonnes clés :")
for col in cols_required_tech:
    present = col in df_tech_final.columns
    print(f"   {'✅' if present else '❌'} {col}")

print("\nWOO — colonnes clés :")
for col in cols_required_woo:
    present = col in df_woo_final.columns
    print(f"   {'✅' if present else '❌'} {col}")

# ── Vérification 2 : cohérence entre les 2 tables ────────────────
# fk_assigned_technician_id dans WOO doit exister dans TECHNICIANS
# (pour l'instant c'est vide donc 0 orphelins attendus)
print("\n── Vérification 2 : cohérence entre les 2 tables ──────────")

tech_ids     = set(df_tech_final["pk_technician_id"].dropna().astype(str))
woo_tech_ids = set(df_woo_final["fk_assigned_technician_id"].dropna().astype(str))
orphans      = woo_tech_ids - tech_ids

print(f"Techniciens dans TECHNICIANS       : {len(tech_ids)}")
print(f"Techniciens référencés dans WOO    : {len(woo_tech_ids)} (vide — normal)")
print(f"Orphelins                          : {len(orphans)} {'✅' if len(orphans)==0 else '❌'}")

#  Vérification 3 : résumé des valeurs manquantes ───────────────
print("\n── Vérification 3 : valeurs manquantes importantes ─────────")

checks = [
    ("WOO — priority_score NULL",        df_woo_final["priority_score"].isna().sum()),
    ("WOO — operation_type NULL",        df_woo_final["operation_type"].isna().sum()),
    ("WOO — required_skill_desc NULL",   df_woo_final["required_skill_desc"].isna().sum()),
    ("WOO — order_basic_start_date NULL",df_woo_final["order_basic_start_date"].isna().sum()),
    ("WOO — asset_description NULL",     df_woo_final["asset_description"].isna().sum()),
    ("TECH — skill_description NULL",    df_tech_final["skill_description"].isna().sum()),
    ("TECH — start_hour NULL",           df_tech_final["start_hour"].isna().sum()),
]

for label, count in checks:
    ok = count == 0
    note = "" if ok else f" ⚠️  {count} valeurs manquantes"
    print(f"   {'✅' if ok else '⚠️ '} {label:<45}{note}")

# ── Vérification 4 : répartition priority_score ──────────────────
print("\n── Vérification 4 : répartition priority_score ─────────────")
labels = {5:"Breakdown", 4:"Meca", 3:"Elec", 2:"monthly", 1:"Low"}
for score, label in labels.items():
    count = len(df_woo_final[df_woo_final["priority_score"] == score])
    barre = "█" * (count // 30)
    print(f"   Score {score} ({label:<10}) → {count:>4} opérations  {barre}")

# ── Vérification 5 : répartition par work_center ─────────────────
print("\n── Vérification 5 : répartition assets par région ──────────")
print(df_woo_final["asset_address_region"].value_counts().to_string())

print("   VÉRIFICATION TERMINÉE — prêt pour sauvegarde CSV")


  VÉRIFICATION FINALE DES 2 TABLES

Table                       Lignes   Colonnes
---------------------------------------------
technicians_final                7         18
woo_final                     1096         28

── Vérification 1 : colonnes clés présentes ────────────────

TECHNICIANS — colonnes clés :
   ✅ pk_technician_id
   ✅ technician_full_name
   ✅ skill_description
   ✅ start_hour
   ✅ end_hour
   ✅ work_center_description

WOO — colonnes clés :
   ✅ pk_woo_id
   ✅ priority_score
   ✅ operation_type
   ✅ operation_subtype
   ✅ required_skill_desc
   ✅ operation_scheduled_start
   ✅ operation_scheduled_end
   ✅ fk_assigned_technician_id

── Vérification 2 : cohérence entre les 2 tables ──────────
Techniciens dans TECHNICIANS       : 7
Techniciens référencés dans WOO    : 0 (vide — normal)
Orphelins                          : 0 ✅

── Vérification 3 : valeurs manquantes importantes ─────────
   ✅ WOO — priority_score NULL                    
   ✅ WOO — operation_type NULL 

Les 2 avertissements sont normaux :
⚠️ order_basic_start_date : 778 NULL → dates manquantes côté client
⚠️ asset_description      : 113 NULL → les 113 WO sans asset identifié


 MODIFICATION priority_score — ordre inversé
 


 1 = priorité maximale (traiter en premier)
 4 = priorité minimale (traiter en dernier)
 Plus naturel pour le Planning Officer et l'algorithme

 Breakdown = 1  ← urgence maximale
 Meca      = 2  ← même priorité que Elec (confirmé client)
 Elec      = 2  ← même priorité que Meca
 monthly   = 3  ← préventif planifiable
 Low       = 4  ← basse priorité
 Inconnu   = NULL ← pas de score attribué

In [8]:


priority_map_new = {
    "Breakdown" : 1,
    "Meca"      : 2,
    "Elec"      : 2,
    "monthly"   : 3,
    "Low"       : 4,
}

df_woo_final["priority_score"] = df_woo_final["operation_subtype"].map(priority_map_new)

# Vérification
print("── Nouveau priority_score ───────────────────────────────────")
labels_new = {1:"Breakdown", 2:"Meca/Elec", 3:"monthly", 4:"Low"}
for score, label in labels_new.items():
    count = len(df_woo_final[df_woo_final["priority_score"] == score])
    barre = "█" * (count // 30)
    print(f"   Score {score} ({label:<12}) → {count:>4} opérations  {barre}")

null_count = df_woo_final["priority_score"].isna().sum()
print(f"   Score NULL (inconnu)  → {null_count:>4} opérations")
print(f"\n priority_score mis à jour")

── Nouveau priority_score ───────────────────────────────────
   Score 1 (Breakdown   ) →  190 opérations  ██████
   Score 2 (Meca/Elec   ) →   63 opérations  ██
   Score 3 (monthly     ) →  220 opérations  ███████
   Score 4 (Low         ) →  623 opérations  ████████████████████
   Score NULL (inconnu)  →    0 opérations

 priority_score mis à jour


 — Sauvegarde des 2 tables finales
 POURQUOI utf-8-sig ?
 Ce format d'encodage ajoute un BOM en début de fichier
 qui permet à Excel d'afficher correctement les accents
 (é, à, ü...) sans avoir à changer les paramètres d'import

In [9]:


# ── Sauvegarde TECHNICIANS 
path_tech = os.path.join(OUTPUT_DIR, "technicians_final.csv")
df_tech_final.to_csv(path_tech, index=False, encoding="utf-8-sig")

# ── Sauvegarde WOO 
path_woo = os.path.join(OUTPUT_DIR, "woo_final.csv")
df_woo_final.to_csv(path_woo, index=False, encoding="utf-8-sig")

# ── Vérification que les fichiers ont bien été créés 
print("── Sauvegarde terminée ")
for path, df, name in [
    (path_tech, df_tech_final, "technicians_final.csv"),
    (path_woo,  df_woo_final,  "woo_final.csv")
]:
    taille = os.path.getsize(path)
    print(f"\n {name}")
    print(f"   Lignes   : {len(df)}")
    print(f"   Colonnes : {len(df.columns)}")
    print(f"   Taille   : {taille / 1024:.1f} KB")
    print(f"   Chemin   : {os.path.abspath(path)}")


print(f"   PHASE fusion  TERMINÉE")
print(f"  2 tables finales prêtes :")
print(f"  → technicians_final.csv  ({len(df_tech_final)} lignes, {len(df_tech_final.columns)} colonnes)")
print(f"  → woo_final.csv          ({len(df_woo_final)} lignes, {len(df_woo_final.columns)} colonnes)")
print(f"══════════════════════════════════════════════════════════════")


── Sauvegarde terminée 

 technicians_final.csv
   Lignes   : 7
   Colonnes : 18
   Taille   : 1.3 KB
   Chemin   : c:\Maria\digitalPlanner\final_tables\technicians_final.csv

 woo_final.csv
   Lignes   : 1096
   Colonnes : 28
   Taille   : 281.0 KB
   Chemin   : c:\Maria\digitalPlanner\final_tables\woo_final.csv
   PHASE fusion  TERMINÉE
  2 tables finales prêtes :
  → technicians_final.csv  (7 lignes, 18 colonnes)
  → woo_final.csv          (1096 lignes, 28 colonnes)
══════════════════════════════════════════════════════════════
